# 01 — Exploración inicial de la base de datos abierta de COVID-19 (DGE)

**Tesis:** Estratificación de riesgo clínico consciente de la equidad — auditoría del sesgo geográfico entre entidades federativas.
**Alcance:** Nacional (32 entidades). **Fuente primaria:** Dirección General de Epidemiología, Secretaría de Salud (México).

Este notebook cubre la Fase 2 del cronograma: **ingesta, diccionario de datos, definición de desenlaces, control de fuga de información, limpieza y análisis exploratorio (EDA)** con énfasis en la dimensión por entidad, que es el eje de equidad de la tesis.

> **Descarga de datos:** obtén el CSV histórico en el portal de Datos Abiertos de la DGE (`https://www.gob.mx/salud/documentos/datos-abiertos-152127`). El archivo se llama típicamente `COVID19MEXICO.csv` (varios GB). Ajusta `DATA_PATH` abajo.


## 1. Configuración e imports

In [1]:
import numpy as np
import pandas as pd

import tesis_common as tc

pd.set_option("display.max_columns", 120)

# Rutas y definición de cohorte viven en tesis_common.py. Los notebooks 02-04
# leen de ahí las mismas constantes, así que no pueden desalinearse de este.
DATA_PATH = tc.CSV_CRUDO
OUT_PARQUET = tc.PARQUET

print("Ruta de datos:", DATA_PATH, "| salida:", OUT_PARQUET)

Ruta de datos: COVID19MEXICO2021.csv | salida: dge_covid2021_limpio.parquet


## 2. Diccionario de datos y catálogos

La base de la DGE codifica las variables categóricas con enteros y usa códigos especiales para valores ausentes:
- **Comorbilidades / síntomas** (`DIABETES`, `HIPERTENSION`, ...): `1 = Sí`, `2 = No`, `97 = No aplica`, `98 = Se ignora`, `99 = No especificado`.
- **`SEXO`**: `1 = Mujer`, `2 = Hombre`, `99 = No especificado`.
- **`TIPO_PACIENTE`**: `1 = Ambulatorio`, `2 = Hospitalizado`, `99 = No especificado`.
- **`FECHA_DEF`**: `9999-99-99` indica paciente **vivo**.
- **`CLASIFICACION_FINAL`**: `1,2,3 = caso COVID confirmado`; `6 = sospechoso`; `7 = negativo`.
- **`ENTIDAD_RES`**: `1..32` entidades; `97/98/99` desconocido.


In [2]:
# Catálogos y listas de variables: definidos una sola vez en tesis_common.py.
ENTIDADES = tc.ENTIDADES
COMORBILIDADES = tc.COMORBILIDADES
VARS_FUGA = tc.VARS_FUGA

print("Entidades:", len(ENTIDADES),
      "| Comorbilidades:", len(COMORBILIDADES),
      "| Variables con riesgo de fuga:", VARS_FUGA)

Entidades: 32 | Comorbilidades: 11 | Variables con riesgo de fuga: ['UCI', 'INTUBADO', 'FECHA_DEF']


## 3. Carga de datos

In [3]:
# La base es grande; se cargan solo las columnas necesarias y con tipos eficientes.
COLS = (["ID_REGISTRO","SEXO","ENTIDAD_RES","MUNICIPIO_RES","EDAD","EMBARAZO","TIPO_PACIENTE",
         "UCI","INTUBADO","CLASIFICACION_FINAL","FECHA_DEF","FECHA_INGRESO","FECHA_SINTOMAS"]
        + COMORBILIDADES)

df = pd.read_csv(DATA_PATH, usecols=lambda c: c in COLS, encoding="latin-1", low_memory=False)
print("Filas x columnas:", df.shape)
df.head(3)

Filas x columnas: (8830345, 24)


,ID_REGISTRO,SEXO,ENTIDAD_RES,MUNICIPIO_RES,TIPO_PACIENTE,FECHA_INGRESO,FECHA_SINTOMAS,FECHA_DEF,INTUBADO,NEUMONIA,EDAD,EMBARAZO,DIABETES,EPOC,ASMA,INMUSUPR,HIPERTENSION,OTRA_COM,CARDIOVASCULAR,OBESIDAD,RENAL_CRONICA,TABAQUISMO,CLASIFICACION_FINAL,UCI
0,z53cb3,2,9,2,1,2021-01-16,2021-01-13,9999-99-99,97,2,26,97,2,2,2,2,2,2,2,2,2,2,7,97
1,zze974,1,24,35,1,2021-02-28,2021-02-20,9999-99-99,97,99,34,2,2,2,2,2,2,2,2,2,2,2,7,97
2,zz7202,2,16,112,1,2021-01-13,2021-01-13,9999-99-99,97,2,41,97,2,2,2,2,2,2,2,2,2,1,7,97


## 4. Definición de desenlaces y control de fuga de información

Se definen los dos desenlaces binarios de la tesis y se separan explícitamente las variables que **no** pueden usarse como predictores porque ocurren *después* del punto de predicción (ingreso).


In [4]:
df["hospitalizado"] = (df["TIPO_PACIENTE"] == 2).astype(int)
df["defuncion"]     = (df["FECHA_DEF"].astype(str) != "9999-99-99").astype(int)

# Se guardan las cifras de la BASE COMPLETA antes de restringir la cohorte: el
# documento contrasta ambas y confundirlas es un error fácil de cometer.
N_BASE_COMPLETA = len(df)
TASA_HOSP_BASE  = float(df["hospitalizado"].mean())
TASA_DEF_BASE   = float(df["defuncion"].mean())

print("Registros en la base completa:", f"{N_BASE_COMPLETA:,}")
print("Tasa de hospitalización:", round(TASA_HOSP_BASE, 4))
print("Tasa de defunción:      ", round(TASA_DEF_BASE, 4))

# Nota: UCI e INTUBADO describen el curso hospitalario -> excluidos como predictores.
# Para el modelo de MORTALIDAD al ingreso, tampoco se usa 'hospitalizado' como predictor.

Registros en la base completa: 8,830,345
Tasa de hospitalización: 0.0641
Tasa de defunción:       0.0206


## 5. Limpieza y recodificación

In [5]:
def recodifica(df):
    """Deriva las variables de análisis a partir de los códigos crudos de la DGE.

    Los cortes de edad y las listas de variables provienen de tesis_common para
    que el notebook 03 no vuelva a construir `grupo_edad` con otros bordes.
    """
    d = df.copy()
    # Edad válida
    d.loc[(d["EDAD"] < 0) | (d["EDAD"] > 120), "EDAD"] = np.nan
    # Sexo (atributo sensible); 1 = Mujer, 2 = Hombre en el catálogo de la DGE
    d["sexo_mujer"] = d["SEXO"].map({1: 1, 2: 0}).astype("Int64")
    # Entidad (eje de equidad geográfica)
    d["entidad_nombre"] = d["ENTIDAD_RES"].map(ENTIDADES)
    # Grupo etario (atributo sensible)
    d["grupo_edad"] = pd.cut(d["EDAD"], bins=tc.BINS_EDAD, labels=tc.ETIQUETAS_EDAD)
    # Comorbilidades: 1->1, 2->0, resto->NaN + indicador de faltante
    for col in COMORBILIDADES:
        d[col + "_na"] = (~d[col].isin([1, 2])).astype(int)
        d[col + "_bin"] = d[col].map({1: 1, 2: 0})
    return d

df = recodifica(df)
df[["EDAD", "grupo_edad", "sexo_mujer", "entidad_nombre", "DIABETES_bin", "DIABETES_na"]].head()

,EDAD,grupo_edad,sexo_mujer,entidad_nombre,DIABETES_bin,DIABETES_na
0,26.0,18-39,0,CIUDAD DE MEXICO,0.0,0
1,34.0,18-39,1,SAN LUIS POTOSI,0.0,0
2,41.0,40-59,0,MICHOACAN,0.0,0
3,25.0,18-39,0,CIUDAD DE MEXICO,0.0,0
4,20.0,18-39,1,CIUDAD DE MEXICO,0.0,0


### 5.1 Filtrado opcional a casos confirmados

Según la pregunta de investigación, puede convenir restringir a casos COVID confirmados (`CLASIFICACION_FINAL in {1,2,3}`). Se deja como bandera para análisis de sensibilidad.


In [6]:
# Cohorte canónica de la tesis, definida en tesis_common: casos COVID confirmados.
# Un modelo de mortalidad por COVID debe estimarse sobre enfermos de COVID;
# incluir negativos y sospechosos mezcla poblaciones y diluye la señal.
SOLO_CONFIRMADOS = tc.SOLO_CONFIRMADOS
if SOLO_CONFIRMADOS:
    df = df[df["CLASIFICACION_FINAL"].isin(tc.CLASIF_CONFIRMADO)].copy()

# Las tasas de la sección 4 se calcularon sobre la base completa; se reportan
# aquí sobre la cohorte de análisis, que es la que alimenta los notebooks 02-04.
print("Filas tras filtrado:", f"{len(df):,}")
print("Tasa de hospitalización (cohorte):", round(df["hospitalizado"].mean(), 4))
print("Tasa de defunción (cohorte):      ", round(df["defuncion"].mean(), 4))

Filas tras filtrado: 2,526,649
Tasa de hospitalización (cohorte): 0.1178
Tasa de defunción (cohorte):       0.056


## 6. Análisis exploratorio (EDA)

In [7]:
# Calidad de datos: faltantes por variable clave
faltantes = df[["EDAD","sexo_mujer","entidad_nombre"]].isna().mean().round(4)
print("Proporción de faltantes:\n", faltantes)
print("\nFaltantes en comorbilidades (código 97/98/99):")
print(df[[c+"_na" for c in COMORBILIDADES]].mean().round(4).sort_values(ascending=False))

Proporción de faltantes:
 EDAD              0.0
sexo_mujer        0.0
entidad_nombre    0.0
dtype: float64

Faltantes en comorbilidades (código 97/98/99):
OTRA_COM_na          0.0088
NEUMONIA_na          0.0057
DIABETES_na          0.0019
EPOC_na              0.0018
INMUSUPR_na          0.0018
HIPERTENSION_na      0.0018
CARDIOVASCULAR_na    0.0018
RENAL_CRONICA_na     0.0018
TABAQUISMO_na        0.0018
ASMA_na              0.0017
OBESIDAD_na          0.0017
dtype: float64


In [8]:
# Distribuciones básicas
print("Distribución por grupo de edad:")
print(df["grupo_edad"].value_counts(dropna=False).sort_index())
print("\nPrevalencia de comorbilidades (entre 1/0 válidos):")
print(df[[c+"_bin" for c in COMORBILIDADES]].mean().round(4).sort_values(ascending=False))

Distribución por grupo de edad:
grupo_edad
0-17      190435
18-39    1212701
40-59     785582
60+       337899
NaN           32
Name: count, dtype: int64

Prevalencia de comorbilidades (entre 1/0 válidos):
HIPERTENSION_bin      0.1184
OBESIDAD_bin          0.0973
DIABETES_bin          0.0900
NEUMONIA_bin          0.0856
TABAQUISMO_bin        0.0598
ASMA_bin              0.0172
OTRA_COM_bin          0.0143
RENAL_CRONICA_bin     0.0093
CARDIOVASCULAR_bin    0.0091
EPOC_bin              0.0067
INMUSUPR_bin          0.0050
dtype: float64


### 6.1 Dimensión geográfica — el eje de equidad

Aquí se calcula el tamaño muestral y las tasas de desenlace por entidad. Una fuerte heterogeneidad entre entidades (en tasas y en tamaño de muestra) es justamente lo que motiva la auditoría de equidad de la tesis: entidades con pocos registros o con perfiles muy distintos son candidatas a que el modelo se desempeñe peor.


In [9]:
por_entidad = (df[df["entidad_nombre"].notna()]
    .groupby("entidad_nombre")
    .agg(n=("hospitalizado","size"),
         tasa_hosp=("hospitalizado","mean"),
         tasa_def=("defuncion","mean"),
         edad_media=("EDAD","mean"))
    .sort_values("n", ascending=False)
    .round(4))
por_entidad

,n,tasa_hosp,tasa_def,edad_media
entidad_nombre,,,,
CIUDAD DE MEXICO,651956,0.0558,0.0262,38.3611
MEXICO,263167,0.1525,0.0717,39.0321
NUEVO LEON,128134,0.1295,0.0527,38.8575
GUANAJUATO,118324,0.1199,0.0574,38.5576
JALISCO,111529,0.1835,0.0896,40.3259
TABASCO,98683,0.0491,0.0228,37.8906
VERACRUZ,80429,0.2043,0.1014,41.1610
PUEBLA,75074,0.2318,0.1111,41.5216
SAN LUIS POTOSI,68653,0.0860,0.0462,39.5495


In [10]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
por_entidad.sort_values("tasa_def")["tasa_def"].plot.barh(ax=ax[0], color="#2E5496")
ax[0].set_title("Tasa de defunción por entidad"); ax[0].set_xlabel("proporción")
por_entidad.sort_values("n")["n"].plot.barh(ax=ax[1], color="#1F3864")
ax[1].set_title("Tamaño de muestra por entidad"); ax[1].set_xlabel("registros")
plt.tight_layout(); plt.savefig("eda_por_entidad.png", dpi=110)
print("Figura guardada: eda_por_entidad.png")

# Exportación de las cifras del EDA que cita el documento. Se exportan en lugar de
# transcribirse: las tasas de la base completa y las de la cohorte confirmada son
# distintas y es fácil citar la equivocada.
# Estructura del desenlace: en la cohorte de confirmados, ¿puede fallecer un
# paciente ambulatorio? La respuesta condiciona como debe leerse el AUROC, porque
# determina en que fraccion de la cohorte el desenlace es siquiera posible.
tab_tipo = pd.crosstab(df["TIPO_PACIENTE"], df["defuncion"])
n_amb_muertos = int(tab_tipo.loc[1, 1]) if (1 in tab_tipo.index and 1 in tab_tipo.columns) else 0
n_hosp = int((df["TIPO_PACIENTE"] == 2).sum())
n_hosp_muertos = int(df.loc[df["TIPO_PACIENTE"] == 2, "defuncion"].sum())
print("\nEstructura del desenlace en la cohorte:")
print(f"  hospitalizados      : {n_hosp:,} ({n_hosp/len(df):.1%} de la cohorte)")
print(f"  defunciones totales : {int(df['defuncion'].sum()):,}")
print(f"  de ellas, ambulatorias: {n_amb_muertos:,}")
if n_amb_muertos == 0:
    print("  -> fallecer implica estar registrado como hospitalizado: el desenlace")
    print("     solo es posible en el subconjunto hospitalizado de la cohorte.")

# ¿Es informativa la codificacion de ausencia? El documento lo afirma para
# justificar que no se impute a ciegas; aqui se comprueba en lugar de asumirlo.
info_na = []
for col in COMORBILIDADES:
    na = df[col + "_na"] == 1
    if na.sum() == 0:
        continue
    info_na.append({"variable": col, "frac_na": float(na.mean()),
                    "mort_con_na": float(df.loc[na, "defuncion"].mean()),
                    "mort_sin_na": float(df.loc[~na, "defuncion"].mean())})
INA = pd.DataFrame(info_na)
INA["razon"] = INA["mort_con_na"] / INA["mort_sin_na"]
print("\nMortalidad segun si la comorbilidad esta codificada como ausente:")
print(INA.sort_values("razon", ascending=False).round(4).to_string(index=False))

# Y su frecuencia, ¿varia entre entidades? Seria un mecanismo de inequidad.
var_na = (df.groupby("entidad_nombre")[[c + "_na" for c in COMORBILIDADES]]
            .mean().max(axis=1))
print(f"\nFrecuencia maxima de faltante por entidad: "
      f"{var_na.min():.4f} ({var_na.idxmin()}) a {var_na.max():.4f} ({var_na.idxmax()})")

na_comorb = df[[c + "_na" for c in COMORBILIDADES]].mean()
calidad = {
    "n_base_completa": int(N_BASE_COMPLETA),
    "n_cohorte": int(len(df)),
    "tasa_def_base_completa": round(float(TASA_DEF_BASE), 6),
    "tasa_hosp_base_completa": round(float(TASA_HOSP_BASE), 6),
    "tasa_def_cohorte": round(float(df["defuncion"].mean()), 6),
    "tasa_hosp_cohorte": round(float(df["hospitalizado"].mean()), 6),
    "faltantes": {
        "EDAD": round(float(df["EDAD"].isna().mean()), 6),
        "sexo_mujer": round(float(df["sexo_mujer"].isna().mean()), 6),
        "entidad_nombre": round(float(df["entidad_nombre"].isna().mean()), 6),
    },
    "na_comorbilidades": {c: round(float(na_comorb[c + "_na"]), 6) for c in COMORBILIDADES},
    "n_hospitalizados": n_hosp,
    "defunciones_totales": int(df["defuncion"].sum()),
    "defunciones_ambulatorias": n_amb_muertos,
    "defunciones_hospitalizadas": n_hosp_muertos,
    "frac_hospitalizados": round(n_hosp / len(df), 6),
    "na_informativo": {
        "razon_max": round(float(INA["razon"].max()), 3),
        "var_razon_max": INA.loc[INA["razon"].idxmax(), "variable"],
        "razon_min": round(float(INA["razon"].min()), 4),
        "var_razon_min": INA.loc[INA["razon"].idxmin(), "variable"],
        "mort_con_na_max": round(float(INA["mort_con_na"].max()), 4),
        "mort_global": round(float(df["defuncion"].mean()), 4),
        "por_variable": INA.round(5).to_dict(orient="records"),
    },
    "na_entidad_min": round(float(var_na.min()), 6),
    "na_entidad_max": round(float(var_na.max()), 6),
    "na_entidad_min_ent": var_na.idxmin(),
    "na_entidad_max_ent": var_na.idxmax(),
    "na_comorb_min": round(float(na_comorb.min()), 6),
    "na_comorb_max": round(float(na_comorb.max()), 6),
    "na_comorb_max_var": na_comorb.idxmax().replace("_na", ""),
    "na_comorb_min_var": na_comorb.idxmin().replace("_na", ""),
    "entidad_n_min": int(por_entidad["n"].min()),
    "entidad_n_max": int(por_entidad["n"].max()),
    "entidad_tasa_min": round(float(por_entidad["tasa_def"].min()), 6),
    "entidad_tasa_max": round(float(por_entidad["tasa_def"].max()), 6),
}
with open("calidad_datos.json", "w", encoding="utf-8") as fh:
    json.dump(calidad, fh, indent=2, ensure_ascii=False)
print("Calidad de datos exportada: calidad_datos.json")

Figura guardada: eda_por_entidad.png

Estructura del desenlace en la cohorte:
  hospitalizados      : 297,685 (11.8% de la cohorte)
  defunciones totales : 141,499
  de ellas, ambulatorias: 0
  -> fallecer implica estar registrado como hospitalizado: el desenlace
     solo es posible en el subconjunto hospitalizado de la cohorte.

Mortalidad segun si la comorbilidad esta codificada como ausente:
      variable  frac_na  mort_con_na  mort_sin_na  razon
          EPOC   0.0018       0.1247       0.0559 2.2314
CARDIOVASCULAR   0.0018       0.1247       0.0559 2.2314
    TABAQUISMO   0.0018       0.1228       0.0559 2.1980
 RENAL_CRONICA   0.0018       0.1222       0.0559 2.1866
          ASMA   0.0017       0.1187       0.0559 2.1244
      INMUSUPR   0.0018       0.1178       0.0559 2.1070
  HIPERTENSION   0.0018       0.1144       0.0559 2.0465
      DIABETES   0.0019       0.1142       0.0559 2.0432
      OBESIDAD   0.0017       0.1103       0.0559 1.9719
      OTRA_COM   0.0088       0


Frecuencia maxima de faltante por entidad: 0.0002 (QUINTANA ROO) a 0.0555 (MEXICO)
Calidad de datos exportada: calidad_datos.json


## 7. Preparación para el modelado (fase siguiente)

Se define la lista de predictores permitidos (sin fuga) y los atributos sensibles para la auditoría de equidad.


In [11]:
PREDICTORES = tc.PREDICTORES
ATRIBUTOS_SENSIBLES = tc.ATRIBUTOS_SENSIBLES  # entidad = eje central

print("Predictores permitidos (sin fuga):", PREDICTORES)
print("Atributos sensibles (auditoría de equidad):", ATRIBUTOS_SENSIBLES)
print("Excluidos por fuga de información:", VARS_FUGA)
print("Punto de operación clínico (sensibilidad objetivo):", tc.SENSIBILIDAD_OBJETIVO)

Predictores permitidos (sin fuga): ['EDAD', 'sexo_mujer', 'DIABETES_bin', 'EPOC_bin', 'ASMA_bin', 'INMUSUPR_bin', 'HIPERTENSION_bin', 'OTRA_COM_bin', 'CARDIOVASCULAR_bin', 'OBESIDAD_bin', 'RENAL_CRONICA_bin', 'TABAQUISMO_bin', 'NEUMONIA_bin']
Atributos sensibles (auditoría de equidad): ['entidad_nombre', 'sexo_mujer', 'grupo_edad']
Excluidos por fuga de información: ['UCI', 'INTUBADO', 'FECHA_DEF']
Punto de operación clínico (sensibilidad objetivo): 0.8


## 8. Guardado del dataset limpio

In [12]:
# Contrato de datos: este parquet es la ÚNICA entrada de los notebooks 02-04.
# El manifiesto registra la cohorte para que aguas abajo se verifique que el
# artefacto corresponde a esta corrida y no a una anterior.
import json
import datetime as dt

cols_guardar = tc.COLUMNAS_PARQUET

ausentes = [c for c in cols_guardar if c not in df.columns]
if ausentes:
    raise KeyError(f"Recodificación incompleta; faltan columnas: {ausentes}")

# Fail-fast deliberado: la versión anterior degradaba a CSV dentro de un `except`,
# lo que dejaba el parquet ANTERIOR en disco y los notebooks 02-04 seguían
# consumiendo otra cohorte sin ningún aviso.
try:
    df[cols_guardar].to_parquet(OUT_PARQUET, index=False)
except ImportError as e:
    raise RuntimeError(
        "Falta el motor de parquet. Instala:  python3 -m pip install 'pyarrow>=14'"
    ) from e

manifiesto = {
    "generado_por": "01_exploracion_datos_dge.ipynb",
    "generado_en": dt.datetime.now().isoformat(timespec="seconds"),
    "fuente": DATA_PATH,
    "solo_confirmados": SOLO_CONFIRMADOS,
    "n_filas": int(len(df)),
    "columnas": cols_guardar,
    "tasa_defuncion": round(float(df["defuncion"].mean()), 6),
    "tasa_hospitalizacion": round(float(df["hospitalizado"].mean()), 6),
    "versiones": {"pandas": pd.__version__, "numpy": np.__version__},
}
with open(tc.MANIFIESTO, "w", encoding="utf-8") as fh:
    json.dump(manifiesto, fh, indent=2, ensure_ascii=False)

print("Guardado:", OUT_PARQUET, "|", len(cols_guardar), "columnas |", f"{len(df):,}", "filas")
print("Manifiesto:", tc.MANIFIESTO, "| solo_confirmados =", SOLO_CONFIRMADOS)

Guardado: dge_covid2021_limpio.parquet | 31 columnas | 2,526,649 filas
Manifiesto: dge_covid2021_limpio.manifest.json | solo_confirmados = True


## 9. Próximos pasos

1. **Modelo de referencia** (regresión logística y gradient boosting) sobre `PREDICTORES` — notebook 02.
2. **Auditoría de equidad** por entidad, sexo y grupo etario (métricas de Fairlearn/AIF360) — notebook 03.
3. **Mitigación de sesgo** y frontera de Pareto equidad–desempeño — notebook 04.
4. **Validación externa** en la base de Egresos Hospitalarios de la DGIS — notebook 05.

> Recordatorio metodológico: nunca usar `UCI`, `INTUBADO` ni información derivada de `FECHA_DEF` como predictores del riesgo al ingreso.
